# ESAM++ Tiny student - all layers trained

Baseline experiment. The published ESAM-E weights are copied into the ESAM++ student wherever the tensors match (103 of 400, all of them in the shared layers), the TinyExpert backbone stays random, and the whole model is trained for 110 minutes on the ESAM-E pseudo instances. This measures what happens when the pretrained shared layers are allowed to drift while the new backbone is still random.

Pipeline on Kaggle (1x T4): restore the ESAM environment, download SceneNN
scene 322, build 80 frames, run the ESAM-E teacher, distil the student,
evaluate and export the point cloud.

Code: `/kaggle/working/tools/distill/` &nbsp;|&nbsp; results: `distill/student_all`


In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/tools/distill /kaggle/working/logs /kaggle/working/distill

cat > /kaggle/working/tools/distill/config.py <<'PYEOF_CONFIG'
"""Paths, data parameters and training parameters of the distillation study.

Every value can be overridden with an environment variable, so the experiment
scripts never need command line parsing.  The experiments themselves live in
`experiments/<name>/settings.json`.
"""
import os
from pathlib import Path

# ============================================================
# 1. LOCATIONS
# ============================================================
WORK = Path(os.environ.get("ESAM_WORK", "/kaggle/working"))
ESAM_ROOT = Path(os.environ.get("ESAM_ROOT", WORK / "ESAM"))
DATA_ROOT = Path(os.environ.get("ESAM_DATA", WORK / "data/scenenn_stream/322"))
RAW_ROOT = Path(os.environ.get("ESAM_RAW", WORK / "tmp/scenenn"))
FRAMES = Path(os.environ.get("ESAM_FRAMES", WORK / "distill/frames"))
TEACHER = Path(os.environ.get("ESAM_TEACHER", WORK / "distill/teacher"))
STUDENT = Path(os.environ.get("ESAM_STUDENT", WORK / "distill/student"))

# ============================================================
# 2. MODEL FILES
# ============================================================
ESAM_CKPT = Path(os.environ.get(
    "ESAM_CKPT", ESAM_ROOT / "work_dirs/ckpt/ESAM-E_CA_online_epoch_128.pth"))
TEACHER_CFG = Path(os.environ.get(
    "ESAM_TEACHER_CFG", ESAM_ROOT / "configs/ESAM-E_CA/ESAM-E_online_stream.py"))
STUDENT_CFG = Path(os.environ.get(
    "ESAM_STUDENT_CFG", WORK / "tools/esampp_tiny_scenenn.py"))

# ============================================================
# 3. DATA PARAMETERS
# ============================================================
NUM_FRAMES = int(os.environ.get("ESAM_NUM_FRAMES", 80))
VAL_START = int(os.environ.get("ESAM_VAL_START", 40))
CLUSTERS = int(os.environ.get("ESAM_CLUSTERS", 100))
SCORE_THR = float(os.environ.get("ESAM_SCORE_THR", 0.3))
MIN_POINTS = int(os.environ.get("ESAM_MIN_POINTS", 100))

# ============================================================
# 4. TRAINING PARAMETERS
# ============================================================
MAX_MINUTES = float(os.environ.get("ESAM_MAX_MINUTES", 110))
LR = float(os.environ.get("ESAM_LR", 1e-3))
DEVICE = "cuda"

# ============================================================
# 5. EXPERIMENT SWITCHES
# ============================================================
FREEZE_SHARED = os.environ.get("ESAM_FREEZE_SHARED", "0") == "1"   # train only the TinyExpert
TARGET = os.environ.get("ESAM_TARGET", "teacher")                  # "teacher" or "gt"
PYEOF_CONFIG

cat > /kaggle/working/tools/distill/common.py <<'PYEOF_COMMON'
"""Geometry, colour, PLY and evaluation helpers shared by the experiment scripts."""
import json
import os

import numpy as np

# ============================================================
# 1. GEOMETRY
# ============================================================
# SceneNN is Y-up, the models expect Z-up: R_x(+90) maps (x, y, z) -> (x, -z, y).
R_ZUP = np.array([[1.0, 0.0, 0.0],
                  [0.0, 0.0, -1.0],
                  [0.0, 1.0, 0.0]])


def zup(points):
    """Rotate SceneNN world points into the Z-up frame of the models."""
    return (np.asarray(points, dtype=np.float64) @ R_ZUP.T).astype(np.float32)


# ============================================================
# 2. COLOURS
# ============================================================
def palette(seed=0, n=256):
    """Deterministic instance colours."""
    rng = np.random.default_rng(seed)
    colors = rng.integers(0, 256, size=(n, 3), dtype=np.uint8)
    colors[-1] = (190, 190, 190)
    return colors


def mask_to_colors(masks, scores=None, colors=None):
    """One stable colour per mask row.  Row order is preserved."""
    masks = np.asarray(masks, dtype=bool)
    colors = palette() if colors is None else colors
    index = np.full(masks.shape[1], -1, dtype=np.int64)
    order = np.argsort(np.asarray(scores)) if scores is not None else range(len(masks))
    for row in order:
        index[masks[row]] = row
    return colors[index]


# ============================================================
# 3. PLY AND POINT CLOUD IO
# ============================================================
def write_ply(path, xyz, rgb):
    """Write a binary little-endian PLY."""
    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)
    header = ("ply\n"
              "format binary_little_endian 1.0\n"
              f"element vertex {len(xyz)}\n"
              "property float x\nproperty float y\nproperty float z\n"
              "property uchar red\nproperty uchar green\nproperty uchar blue\n"
              "end_header\n")
    record = np.empty(len(xyz), dtype=[("x", "<f4"), ("y", "<f4"), ("z", "<f4"),
                                       ("red", "u1"), ("green", "u1"), ("blue", "u1")])
    record["x"], record["y"], record["z"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    record["red"], record["green"], record["blue"] = rgb[:, 0], rgb[:, 1], rgb[:, 2]
    with open(path, "wb") as f:
        f.write(header.encode("ascii"))
        record.tofile(f)


def voxel_downsample(xyz, rgb, voxel=0.02):
    """Keep one point per occupied voxel."""
    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)
    if len(xyz) == 0:
        return xyz, rgb
    keys = np.floor((xyz - xyz.min(axis=0)) / voxel).astype(np.int64)
    keys = np.clip(keys, 0, (1 << 21) - 1)
    packed = (keys[:, 0] << 42) | (keys[:, 1] << 21) | keys[:, 2]
    keep = np.sort(np.unique(packed, return_index=True)[1])
    return xyz[keep], rgb[keep]


def save_preview(path, xyz, rgb, title=""):
    """Save a two-view matplotlib preview of a point cloud."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)
    if len(xyz) > 90000:
        sel = np.random.default_rng(0).choice(len(xyz), 90000, replace=False)
        xyz, rgb = xyz[sel], rgb[sel]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), dpi=140)
    for ax, (name, a, b) in zip(axes, [("X / Z", 0, 2), ("X / Y", 0, 1)]):
        ax.scatter(xyz[:, a], xyz[:, b], c=rgb / 255.0, s=0.25, linewidths=0)
        ax.set_title(name)
        ax.set_aspect("equal", adjustable="box")
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 4. EVALUATION
# ============================================================
def iou_matrix(pred, gt):
    """IoU between predicted masks [P, N] and ground truth masks [G, N]."""
    pred = np.asarray(pred, dtype=np.float32)
    gt = np.asarray(gt, dtype=np.float32)
    if pred.size == 0 or gt.size == 0:
        return np.zeros((len(pred), len(gt)), dtype=np.float32)
    inter = pred @ gt.T
    union = pred.sum(1)[:, None] + gt.sum(1)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-6), 0.0)


def average_precision(pred_masks, pred_scores, gt_masks, threshold=0.5):
    """Class agnostic AP at one IoU threshold.  gt_masks: [G, N] bool."""
    if len(gt_masks) == 0 or len(pred_masks) == 0:
        return 0.0
    ious = iou_matrix(pred_masks, gt_masks)
    order = np.argsort(-np.asarray(pred_scores))
    used = np.zeros(len(gt_masks), dtype=bool)
    tp = np.zeros(len(order), dtype=np.float32)
    for rank, row in enumerate(order):
        best = int(np.argmax(ious[row]))
        if ious[row, best] >= threshold and not used[best]:
            used[best] = True
            tp[rank] = 1.0
    if tp.sum() == 0:
        return 0.0
    tp_c = np.cumsum(tp)
    fp_c = np.cumsum(1.0 - tp)
    recall = tp_c / len(gt_masks)
    precision = tp_c / np.maximum(tp_c + fp_c, 1e-6)
    recall = np.concatenate([[0.0], recall, [1.0]])
    precision = np.concatenate([[0.0], precision, [0.0]])
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])
    idx = np.where(recall[1:] != recall[:-1])[0]
    return float(np.sum((recall[idx + 1] - recall[idx]) * precision[idx + 1]))


def instance_ap(pred_masks, pred_scores, gt_ins, min_gt_points=50):
    """AP averaged over IoU 0.50:0.95, plus AP50 and AP25, for one frame."""
    pred_masks = np.asarray(pred_masks, dtype=bool)
    pred_scores = np.asarray(pred_scores, dtype=np.float32)
    gt_ins = np.asarray(gt_ins, dtype=np.int64)
    if len(pred_masks) > 300:
        top = np.argsort(-pred_scores)[:300]
        pred_masks, pred_scores = pred_masks[top], pred_scores[top]
    gt_ids = [x for x in np.unique(gt_ins) if x > 0]
    gt_masks = [gt_ins == gid for gid in gt_ids
                if (gt_ins == gid).sum() >= min_gt_points]
    if not gt_masks:
        return {"AP": 0.0, "AP50": 0.0, "AP25": 0.0,
                "num_pred": int(len(pred_masks)), "num_gt": 0}
    gt_masks = np.stack(gt_masks)
    thresholds = list(np.arange(0.50, 0.96, 0.05))
    values = [average_precision(pred_masks, pred_scores, gt_masks, t) for t in thresholds]
    return {"AP": float(np.mean(values)),
            "AP50": float(average_precision(pred_masks, pred_scores, gt_masks, 0.50)),
            "AP25": float(average_precision(pred_masks, pred_scores, gt_masks, 0.25)),
            "num_pred": int(len(pred_masks)), "num_gt": int(len(gt_masks))}


# ============================================================
# 5. JSON IO
# ============================================================
def write_json(path, data):
    """Write a JSON report file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=float)
PYEOF_COMMON

cat > /kaggle/working/tools/distill/data.py <<'PYEOF_DATA'
"""Build the SceneNN 322 frames used by every experiment.

Each frame holds the 20 000 points of the scene in world Z-up coordinates, the
super point grouping and the exact per-point instance ids.
"""
import numpy as np
from sklearn.cluster import KMeans

import config as C
from common import write_json, zup

# ImageNet statistics used by the ESAM models for the colour channels.
RGB_MEAN = np.array([0.47793125906962, 0.4303257521323044, 0.3749598901421883])
RGB_STD = np.array([0.2294476613552524, 0.22136130327957347, 0.22412925035485958])


# ============================================================
# 1. FRAME LOADING
# ============================================================
def load_frame(frame_id):
    """Load one SceneNN frame: points, super points, instance ids, colours."""
    name = f"{frame_id:05d}.npy"
    scene = C.RAW_ROOT / "oni" / "322"
    point = np.load(scene / "point" / name)
    ins = np.load(scene / "ins" / name)
    pose = np.load(scene / "pose" / name)

    xyz = point[:, :3]
    xyz = (pose @ np.c_[xyz, np.ones(len(xyz))].T).T[:, :3]
    xyz = zup(xyz)

    rgb = point[:, 3:6].astype(np.float32)
    norm_rgb = (rgb / 255.0 - RGB_MEAN) / RGB_STD
    super_points = KMeans(n_clusters=C.CLUSTERS, n_init=10,
                           random_state=0).fit_predict(xyz)
    return (np.c_[xyz, norm_rgb].astype(np.float32),
            super_points.astype(np.int64),
            ins.astype(np.int64),
            rgb.astype(np.uint8))


def frame_ids():
    """Frame ids of the study: the first NUM_FRAMES ids of the converted stream."""
    ids = [int(x) for x in (C.DATA_ROOT / "frame_ids.txt").read_text().split()]
    return ids[:C.NUM_FRAMES]


# ============================================================
# 2. MAIN
# ============================================================
def main():
    """Write one npz per frame plus a small meta file."""
    C.FRAMES.mkdir(parents=True, exist_ok=True)
    ids = frame_ids()
    for index, frame_id in enumerate(ids):
        points, super_points, ins, rgb = load_frame(frame_id)
        np.savez_compressed(C.FRAMES / f"frame_{index:04d}.npz",
                            points=points, sp_pts_mask=super_points,
                            gt_ins=ins, rgb_raw=rgb,
                            frame_id=np.array([frame_id]))
        if (index + 1) % 20 == 0:
            print(f"prepared {index + 1}/{len(ids)} frames")
    write_json(C.FRAMES / "meta.json",
               {"frames": len(ids), "frame_ids": ids,
                "points_per_frame": 20000, "clusters": C.CLUSTERS,
                "rotation": "R_x(+90) Y-up -> Z-up"})
    print("frames ready:", C.FRAMES)


if __name__ == "__main__":
    main()
PYEOF_DATA

cat > /kaggle/working/tools/distill/teacher.py <<'PYEOF_TEACHER'
"""Run the ESAM-E online teacher over the prepared frames.

The teacher produces the pseudo instances that the student imitates, and the
reference AP that every experiment is measured against.
"""
import glob
import sys
import time

import numpy as np
import torch

import config as C
from common import (instance_ap, mask_to_colors, palette, save_preview,
                    voxel_downsample, write_json, write_ply)

sys.path.insert(0, str(C.ESAM_ROOT))


# ============================================================
# 1. TEACHER MODEL
# ============================================================
def build_teacher():
    """Build ESAM-E with the published CA online checkpoint."""
    import oneformer3d  # noqa: F401
    from mmengine.config import Config
    from mmengine.registry import init_default_scope
    from mmengine.runner import load_checkpoint
    from mmdet3d.registry import MODELS

    cfg = Config.fromfile(C.TEACHER_CFG)
    init_default_scope(cfg.get("default_scope", "mmdet3d"))
    model = MODELS.build(cfg.model)
    load_checkpoint(model, str(C.ESAM_CKPT), map_location="cpu")
    model.cfg = cfg
    model.map_to_rec_pcd = False
    return model.to(C.DEVICE).eval()


def teacher_frame(model, frame):
    """Predict the instances of one frame with the online ESAM-E model."""
    from mmengine.dataset import pseudo_collate
    from mmdet3d.structures import Det3DDataSample, PointData

    points = torch.from_numpy(frame["points"]).float().to(C.DEVICE)
    sample = Det3DDataSample()
    gt_pts = PointData()
    gt_pts["sp_pts_mask"] = torch.from_numpy(frame["sp_pts_mask"]).long().to(C.DEVICE)
    sample.gt_pts_seg = gt_pts
    data = pseudo_collate([dict(inputs={"points": points}, data_samples=sample)])
    with torch.no_grad():
        result = model.test_step(data)
    pred = result[0].pred_pts_seg
    return (np.asarray(pred.pts_instance_mask[0], dtype=bool),
            np.asarray(pred.instance_scores, dtype=np.float32))


# ============================================================
# 2. MAIN
# ============================================================
def main():
    """Predict all frames, store the pseudo instances and write the metrics."""
    files = sorted(glob.glob(str(C.FRAMES / "frame_*.npz")))
    C.TEACHER.mkdir(parents=True, exist_ok=True)
    colors = palette()
    model = build_teacher()

    points_all, gt_all, counts = [], [], []
    frame_ap, stats = [], []
    total = 0
    start = time.time()

    for index, path in enumerate(files):
        data = np.load(path)
        points = data["points"].astype(np.float32)
        gt = data["gt_ins"].astype(np.int64)
        num_points = len(points)

        merged, scores = teacher_frame(model, {k: data[k] for k in data.files})
        current = merged[:, total:total + num_points]
        frame_ap.append(instance_ap(current, scores, gt))

        keep = [i for i in range(len(current))
                if current[i].sum() >= C.MIN_POINTS and scores[i] >= C.SCORE_THR]
        masks = current[keep] if keep else np.zeros((0, num_points), dtype=bool)
        mask_scores = scores[keep] if keep else np.zeros((0,), dtype=np.float32)
        np.savez_compressed(C.TEACHER / f"pseudo_{index:04d}.npz",
                            masks=masks, scores=mask_scores,
                            frame_id=data["frame_id"])

        points_all.append(points[:, :3])
        gt_all.append(gt)
        counts.append(len(masks))
        stats.append({"index": index, "frame_id": int(data["frame_id"][0]),
                      "instances": int(len(masks)),
                      "max_mask_points": int(masks.sum(axis=1).max()) if len(masks) else 0})
        total += num_points

        if index < 4:
            save_preview(C.TEACHER / f"teacher_preview_{index:04d}.png",
                         points[:, :3], mask_to_colors(masks, mask_scores, colors),
                         title=f"ESAM-E teacher frame {index} ({len(masks)} instances)")
        if (index + 1) % 20 == 0:
            print(f"teacher frame {index + 1}/{len(files)}")

    np.savez_compressed(C.TEACHER / "teacher_final.npz",
                        points=np.concatenate(points_all),
                        gt_ins=np.concatenate(gt_all),
                        instances=np.array(counts))

    val_pts = np.concatenate(points_all[C.VAL_START:])
    val_offset = sum(len(p) for p in points_all[:C.VAL_START])
    val_masks = merged[:, val_offset:]
    val_colors = mask_to_colors(val_masks, scores, colors)
    write_ply(C.TEACHER / "teacher_val.ply", val_pts, val_colors)
    down_xyz, down_colors = voxel_downsample(val_pts, val_colors, 0.02)
    write_ply(C.TEACHER / "teacher_val_voxel02.ply", down_xyz, down_colors)

    # ============================================================
    # 3. METRICS AND SANITY GATE
    # ============================================================
    def mean_ap(selection):
        chosen = [a for i, a in enumerate(frame_ap)
                  if selection(i) and a["num_gt"] > 0]
        if not chosen:
            return {"AP": 0.0, "AP50": 0.0, "AP25": 0.0, "frames": 0}
        return {"AP": float(np.mean([a["AP"] for a in chosen])),
                "AP50": float(np.mean([a["AP50"] for a in chosen])),
                "AP25": float(np.mean([a["AP25"] for a in chosen])),
                "frames": len(chosen)}

    ap_val = mean_ap(lambda i: i >= C.VAL_START)
    ap_all = mean_ap(lambda i: True)
    fractions = [s["max_mask_points"] / 20000.0 for s in stats if s["max_mask_points"]]
    gate = {"ok": bool(len(files) >= 20 and np.median(counts) >= 2
                       and 0.01 <= np.median(fractions) <= 0.95),
            "frames": len(files),
            "median_instances": float(np.median(counts)),
            "median_max_mask_fraction": float(np.median(fractions))}
    write_json(C.TEACHER / "teacher_metrics.json",
               {"frames": len(files), "elapsed_sec": time.time() - start,
                "teacher_ap_val": ap_val, "teacher_ap_all": ap_all,
                "gate": gate, "frame_stats": stats})
    print("teacher AP (val):", ap_val)
    print("gate:", gate)


if __name__ == "__main__":
    main()
PYEOF_TEACHER

cat > /kaggle/working/tools/distill/tiny_model.py <<'PYEOF_TINY'
"""ESAM++ Tiny student model.

ESAM++ keeps every layer of ESAM except the 3D encoder: the heavy sparse
U-Net is replaced by the TinyExpert, a small hierarchical sparse feature
pyramid.  This module registers the student model without touching any
upstream ESAM file.
"""
import os
import sys

import MinkowskiEngine as ME
import torch
from mmdet3d.registry import MODELS
from oneformer3d.mixformer3d import ScanNet200MixFormer3D_Stream

# The notebook copies google/esamplusplus/src to tools/esampp_src.
HERE = os.path.dirname(os.path.abspath(__file__))
TOOLS = os.path.dirname(HERE)
for path in (TOOLS, HERE):
    if path not in sys.path:
        sys.path.insert(0, path)

from esampp_src.backbone import TinyExpert  # noqa: F401,E402  (registers the backbone)


# ============================================================
# 1. STUDENT MODEL
# ============================================================
@MODELS.register_module()
class ScanNet200MixFormer3D_Online_Tiny(ScanNet200MixFormer3D_Stream):
    """Single frame online variant whose backbone takes no `memory` kwarg."""

    def extract_feat(self, batch_inputs_dict, batch_data_samples):
        """Replace the heavy sparse U-Net by the TinyExpert, keep the rest."""
        coordinates, features = [], []
        for points in batch_inputs_dict["points"]:
            coordinates.append(points[:, :3])
            features.append(points[:, 3:])
        all_xyz = coordinates

        coordinates, features = ME.utils.batch_sparse_collate(
            [(c / self.voxel_size, f) for c, f in zip(coordinates, features)],
            device=coordinates[0].device)
        field = ME.TensorField(coordinates=coordinates, features=features)

        x = self.backbone(field.sparse())          # TinyExpert has no memory
        x = x.slice(field)
        point_features = [torch.cat([c, f], dim=-1)
                          for c, f in zip(all_xyz, x.decomposed_features)]
        x = x.features

        sp_pts_masks, n_super_points = [], []
        for data_sample in batch_data_samples:
            sp_pts_mask = data_sample.gt_pts_seg.sp_pts_mask
            sp_pts_masks.append(sp_pts_mask + sum(n_super_points))
            n_super_points.append(sp_pts_mask.max() + 1)
        x, all_xyz_w = self.pool(x, torch.cat(sp_pts_masks), all_xyz, with_xyz=True)

        features, sp_xyz_list = [], []
        for i in range(len(n_super_points)):
            begin = sum(n_super_points[:i])
            end = sum(n_super_points[:i + 1])
            features.append(x[begin:end, :-3])
            sp_xyz_list.append(x[begin:end, -3:])
        return features, point_features, all_xyz_w, sp_xyz_list, None, None
PYEOF_TINY

cat > /kaggle/working/tools/distill/student.py <<'PYEOF_STUDENT'
"""Train the ESAM++ Tiny student to imitate the ESAM-E teacher instances.

The experiment switches in `config.py` decide what is trained:

* `ESAM_FREEZE_SHARED=1` freezes every shared ESAM++ layer and trains only the
  TinyExpert backbone (the official ESAM++ recipe).
* `ESAM_FREEZE_SHARED=0` trains the whole model.
* `ESAM_TARGET=teacher` imitates the ESAM-E pseudo instances, `gt` uses the
  exact SceneNN instance ids.
"""
import glob
import os
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment

import config as C
from common import (instance_ap, mask_to_colors, palette, save_preview,
                    voxel_downsample, write_json, write_ply)

HERE = os.path.dirname(os.path.abspath(__file__))
for path in (str(C.ESAM_ROOT), HERE, os.path.dirname(HERE)):
    if path not in sys.path:
        sys.path.insert(0, path)


# ============================================================
# 1. DATA
# ============================================================
def load_frame(path):
    """Points, super points, instance ids and raw colours of one frame."""
    data = np.load(path)
    return (torch.from_numpy(data["points"]).float(),
            torch.from_numpy(data["sp_pts_mask"]).long(),
            data["gt_ins"].astype(np.int64),
            data["rgb_raw"].astype(np.uint8))


def load_pseudo(path):
    """Teacher instances of one frame: masks and scores."""
    data = np.load(path)
    return torch.from_numpy(data["masks"]).bool(), torch.from_numpy(data["scores"]).float()


def load_gt_targets(path, min_points=50):
    """Instance masks built from the ground truth ids of one frame."""
    ids = np.load(path)["gt_ins"].astype(np.int64)
    masks = [ids == gid for gid in np.unique(ids)
             if gid > 0 and (ids == gid).sum() >= min_points]
    if not masks:
        return torch.zeros((0, len(ids)), dtype=torch.bool), torch.zeros(0)
    return torch.from_numpy(np.stack(masks)), torch.ones(len(masks))


def build_cache(frame_paths, pseudo_paths):
    """Cache the frames and their targets in RAM."""
    cache = []
    for index, frame_path in enumerate(frame_paths):
        points, sp, _, _ = load_frame(frame_path)
        masks, scores = (load_gt_targets(frame_path) if C.TARGET == "gt"
                         else load_pseudo(pseudo_paths[index]))
        cache.append((points, sp, masks, scores))
    return cache


# ============================================================
# 2. MODEL
# ============================================================
def build_student():
    """Build the ESAM++ Tiny student from the ESAM++ config."""
    import oneformer3d  # noqa: F401
    import tiny_model  # noqa: F401
    from mmengine.config import Config
    from mmengine.registry import init_default_scope
    from mmdet3d.registry import MODELS

    cfg = Config.fromfile(C.STUDENT_CFG)
    cfg.custom_imports = dict(imports=["oneformer3d", "tiny_model"])
    init_default_scope(cfg.get("default_scope", "mmdet3d"))
    model = MODELS.build(cfg.model)
    model.cfg = cfg
    return model.to(C.DEVICE)


def partial_init(model):
    """Copy the published ESAM-E weights that still fit the student."""
    checkpoint = torch.load(str(C.ESAM_CKPT), map_location="cpu")
    source = checkpoint.get("state_dict", checkpoint)
    source = {k.replace("module.", "", 1): v for k, v in source.items()}
    own = model.state_dict()
    matched = {k: v for k, v in source.items()
               if k in own and own[k].shape == v.shape}
    model.load_state_dict(matched, strict=False)
    return {"matched": len(matched), "total": len(own)}


def set_trainable(model, freeze_shared):
    """Freeze the shared layers, train only the TinyExpert backbone."""
    for name, parameter in model.named_parameters():
        parameter.requires_grad = (not freeze_shared) or name.startswith("backbone.")
    return [p for p in model.parameters() if p.requires_grad]


def set_train_mode(model):
    """Backbone in train mode, the frozen shared layers stay in eval mode."""
    model.train()
    if C.FREEZE_SHARED:
        for name, module in model.named_modules():
            if not name.startswith("backbone"):
                module.eval()


def student_forward(model, points, sp):
    """Student queries: mask logits and objectness logits."""
    from mmdet3d.structures import Det3DDataSample, PointData

    sample = Det3DDataSample()
    gt_pts = PointData()
    gt_pts["sp_pts_mask"] = sp
    sample.gt_pts_seg = gt_pts
    x, point_features, all_xyz, sp_xyz, _, _ = model.extract_feat(
        {"points": [points]}, [sample])
    out = model.decoder(x, point_features, x, ([sp], all_xyz))
    masks, cls = out["masks"][0], out["cls_preds"][0]
    if isinstance(masks, (list, tuple)):
        masks = masks[-1]
    if isinstance(cls, (list, tuple)):
        cls = cls[-1]
    return masks, cls


# ============================================================
# 3. LOSS
# ============================================================
def distill_loss(logits, cls, target_masks, target_scores):
    """Hungarian matching between student queries and target instances."""
    if len(target_masks) == 0:
        return F.binary_cross_entropy_with_logits(
            cls[:, 1], torch.zeros_like(cls[:, 1])) * 0.1
    prob = logits.sigmoid().detach()
    target = target_masks.float()
    inter = prob @ target.T
    union = prob.sum(1)[:, None] + target.sum(1)[None, :] - inter
    iou = inter / union.clamp_min(1e-6)
    rows, cols = linear_sum_assignment((1.0 - iou).detach().cpu().numpy())
    rows = torch.as_tensor(rows, device=logits.device)
    cols = torch.as_tensor(cols, device=logits.device)

    chosen, wanted = logits[rows], target[cols]
    weight = target_scores[cols].clamp(0.05, 1.0)
    bce = F.binary_cross_entropy_with_logits(
        chosen, wanted, reduction="none").mean(dim=1)
    bce = (bce * weight).sum() / weight.sum().clamp_min(1e-6)
    prob_chosen = chosen.sigmoid()
    dice = (2 * (prob_chosen * wanted).sum(dim=1) + 1) / (
        prob_chosen.sum(dim=1) + wanted.sum(dim=1) + 1)
    dice = (1 - dice).mean()
    objectness = torch.zeros_like(cls[:, 1])
    objectness[rows] = 1.0
    cls_loss = F.binary_cross_entropy_with_logits(
        cls[:, 1], objectness, pos_weight=torch.tensor(2.0, device=cls.device))
    return bce + dice + 0.5 * cls_loss


# ============================================================
# 4. EVALUATION
# ============================================================
def evaluate(model, frames):
    """Class agnostic AP of the student, averaged over the given frames."""
    from mmengine.dataset import pseudo_collate
    from mmdet3d.structures import Det3DDataSample, PointData

    if hasattr(model, "online_merger"):
        delattr(model, "online_merger")
    model.eval()
    prev = 0
    frame_ap = []
    xyz_parts, rgb_parts, gt_parts = [], [], []
    merged = np.zeros((0, 0), dtype=bool)
    scores = np.zeros((0,), dtype=np.float32)
    with torch.no_grad():
        for path in frames:
            points, sp, gt, rgb = load_frame(path)
            num_points = len(points)
            sample = Det3DDataSample()
            gt_pts = PointData()
            gt_pts["sp_pts_mask"] = sp.to(C.DEVICE)
            sample.gt_pts_seg = gt_pts
            data = pseudo_collate([dict(
                inputs={"points": points.to(C.DEVICE)}, data_samples=sample)])
            result = model.test_step(data)
            pred = result[0].pred_pts_seg
            merged = np.asarray(pred.pts_instance_mask[0], dtype=bool)
            scores = np.asarray(pred.instance_scores, dtype=np.float32)
            frame_ap.append(instance_ap(merged[:, prev:prev + num_points], scores, gt))
            xyz_parts.append(points[:, :3].numpy())
            rgb_parts.append(rgb)
            gt_parts.append(gt)
            prev += num_points

    chosen = [a for a in frame_ap if a["num_gt"] > 0]
    ap = {"AP": float(np.mean([a["AP"] for a in chosen])),
          "AP50": float(np.mean([a["AP50"] for a in chosen])),
          "AP25": float(np.mean([a["AP25"] for a in chosen])),
          "num_pred": int(len(scores)),
          "num_gt": int(sum(a["num_gt"] for a in chosen)),
          "frames": len(chosen)}
    keep = [i for i in range(len(merged)) if merged[i].any()]
    masks = merged[keep] if keep else np.zeros((0, prev), dtype=bool)
    keep_scores = scores[keep] if keep else scores
    gt = np.concatenate(gt_parts)
    xyz = np.concatenate(xyz_parts)
    rgb = np.concatenate(rgb_parts)
    return ap, masks, keep_scores, xyz, rgb, gt


# ============================================================
# 5. TRAINING
# ============================================================
def train(model, train_cache, val_cache, minutes):
    """Train the student for the given budget and return the best checkpoint."""
    trainable = set_trainable(model, C.FREEZE_SHARED)
    optimizer = torch.optim.AdamW(trainable, lr=C.LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(200, int(minutes * 12)), eta_min=C.LR * 0.05)
    best_path = C.STUDENT / "student_best.pth"
    last_path = C.STUDENT / "student_last.pth"
    history, best_val, step = [], float("inf"), 0
    start = time.time()

    def save(path):
        torch.save({"state_dict": model.state_dict(), "step": step}, path)

    def val_loss():
        model.eval()
        values = []
        with torch.no_grad():
            for points, sp, masks, scores in val_cache[:5]:
                logits, cls = student_forward(model, points.to(C.DEVICE), sp.to(C.DEVICE))
                values.append(float(distill_loss(logits, cls,
                                                 masks.to(C.DEVICE),
                                                 scores.to(C.DEVICE)).item()))
        set_train_mode(model)
        return float(np.mean(values)) if values else 0.0

    set_train_mode(model)
    while time.time() - start < minutes * 60:
        points, sp, masks, scores = train_cache[step % len(train_cache)]
        logits, cls = student_forward(model, points.to(C.DEVICE), sp.to(C.DEVICE))
        loss = distill_loss(logits, cls, masks.to(C.DEVICE), scores.to(C.DEVICE))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        step += 1
        if step % 20 == 0:
            history.append({"step": step, "loss": float(loss.item())})
            print(f"step {step} loss {float(loss.item()):.4f} "
                  f"{(time.time() - start) / 60:.1f}m")
        if step % 200 == 0:
            value = val_loss()
            history.append({"step": step, "val_loss": value})
            print(f"val step {step} loss {value:.4f}")
            if value < best_val:
                best_val = value
                save(best_path)
        if step % 100 == 0:
            save(last_path)

    save(last_path)
    if not best_path.exists():
        save(best_path)
    return best_path, {"steps": step, "train_minutes": (time.time() - start) / 60,
                        "best_val_loss": best_val, "history": history,
                        "trainable_tensors": len(trainable)}


# ============================================================
# 6. MAIN
# ============================================================
def main():
    """Run one experiment: prepare, train, evaluate and export."""
    frame_files = sorted(glob.glob(str(C.FRAMES / "frame_*.npz")))
    pseudo_files = sorted(glob.glob(str(C.TEACHER / "pseudo_*.npz")))
    train_frames = frame_files[:C.VAL_START]
    val_frames = frame_files[C.VAL_START:]

    C.STUDENT.mkdir(parents=True, exist_ok=True)
    model = build_student()
    init_info = partial_init(model)
    train_cache = build_cache(train_frames, pseudo_files[:C.VAL_START])
    val_cache = build_cache(val_frames, pseudo_files[C.VAL_START:])

    points, sp, masks, scores = train_cache[0]
    logits, cls = student_forward(model, points.to(C.DEVICE), sp.to(C.DEVICE))
    print("student forward:", tuple(logits.shape), tuple(cls.shape))

    best_path, summary = train(model, train_cache, val_cache, C.MAX_MINUTES)
    model.load_state_dict(torch.load(str(best_path), map_location="cpu")["state_dict"])

    ap, masks, scores, xyz, rgb, gt = evaluate(model, val_frames)
    colors = mask_to_colors(masks, scores, palette())
    write_ply(C.STUDENT / "student_val.ply", xyz, colors)
    down_xyz, down_colors = voxel_downsample(xyz, colors, 0.02)
    write_ply(C.STUDENT / "student_val_voxel02.ply", down_xyz, down_colors)
    save_preview(C.STUDENT / "student_preview.png", xyz, colors, title=f"Tiny student {ap}")
    write_json(C.STUDENT / "student_ap.json", ap)
    write_json(C.STUDENT / "distill_metrics.json",
               {**summary, "student_ap": ap, "init_info": init_info,
                "frozen_shared": C.FREEZE_SHARED, "target": C.TARGET,
                "train_frames": len(train_frames), "val_frames": len(val_frames)})
    print("student AP (val):", ap)


if __name__ == "__main__":
    main()
PYEOF_STUDENT



In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
SRC=/kaggle/input/notebooks/abdesselamaimen01/notebook8491230ab9
mkdir -p "$W/logs" "$W/.state" "$W/tools"
echo "restoring saved environment"
tar -xzf "$SRC/esam_env.tar.gz" -C "$W" --ignore-zeros \
  2>&1 | tee "$W/logs/00_restore.log" || true
for d in ESAM FastSAM data; do
  rm -rf "$W/$d"
  cp -a "$SRC/$d" "$W/"
done
cp "$SRC/setup.sh" "$W/setup.sh" 2>/dev/null || true
export PATH="$W/mc/envs/ESAM/bin:$PATH"
export LD_LIBRARY_PATH="$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}"
export PYTHONPATH="$W/ESAM:$W/FastSAM"
env_ok=1
$W/mc/envs/ESAM/bin/python - <<'PY' || env_ok=0
import torch, MinkowskiEngine, mmdet3d, pointops
print('torch', torch.__version__, '| ME', MinkowskiEngine.__version__,
      '| cuda', torch.cuda.is_available())
PY
if [ "$env_ok" -eq 0 ]; then
  echo "saved env incomplete; rebuilding with setup.sh"
  rm -rf "$W/mc"
  for s in esam_repo fastsam esam_ckpt scenenn scene_convert; do
    touch "$W/.state/$s.ok"
  done
  bash "$W/setup.sh" 2>&1 | tee "$W/logs/setup_fallback.log"
  grep -q 'ALL_STEPS_OK' "$W/.state/setup_status.txt"
  export PATH="$W/mc/envs/ESAM/bin:$PATH"
  export LD_LIBRARY_PATH="$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}"
  export PYTHONPATH="$W/ESAM:$W/FastSAM"
fi
echo "$W/FastSAM" > "$W/mc/envs/ESAM/lib/python3.8/site-packages/fastsam_path.pth"
$W/mc/envs/ESAM/bin/python - <<'PY'
import torch, MinkowskiEngine, mmdet3d, pointops
print('ready:', torch.__version__, MinkowskiEngine.__version__,
      'cuda', torch.cuda.is_available())
PY
df -h "$W" | tail -1


In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
if [ ! -d "$W/esamplusplus/.git" ]; then
  git clone --depth 1 https://github.com/google/esamplusplus.git "$W/esamplusplus"
fi
rm -rf "$W/tools/esampp_src"
cp -a "$W/esamplusplus/src" "$W/tools/esampp_src"
touch "$W/tools/esampp_src/__init__.py"
cp "$W/esamplusplus/configs/esamplusplus_online_scannet200_CA.py" "$W/tools/esampp_tiny_scenenn.py"
ls -lh "$W/tools/esampp_src" "$W/tools/esampp_tiny_scenenn.py"


In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
mkdir -p "$W/tmp/scenenn" "$W/distill/frames" "$W/logs"
URL="https://huggingface.co/XXXCARREY/EmbodiedSAM/resolve/main/SceneNN/SceneNN.tar.0?download=true"
echo "streaming SceneNN part 0: point, ins, pose"
curl -fL --retry 3 "$URL" | tar -xf - -C "$W/tmp/scenenn" --ignore-zeros \
  --wildcards 'oni/322/point/*' 'oni/322/ins/*' 'oni/322/pose/*' \
  2>&1 | tee "$W/logs/10_extract_npy.log" || true
P=$(find "$W/tmp/scenenn" -path '*/322/point/*.npy' | wc -l)
I=$(find "$W/tmp/scenenn" -path '*/322/ins/*.npy' | wc -l)
echo "point/ins npy count: $P / $I"
[ "$P" -ge 80 ] && [ "$I" -ge 80 ]
df -h "$W" | tail -1


In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
export PATH=$W/mc/envs/ESAM/bin:$PATH
export LD_LIBRARY_PATH=$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}
export CUDA_VISIBLE_DEVICES=0
cd $W
$W/mc/envs/ESAM/bin/python $W/tools/distill/data.py \
  2>&1 | tee $W/logs/20_prepare.log
ls -lh $W/distill/frames | tail -3
df -h $W | tail -1


In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
export PATH=$W/mc/envs/ESAM/bin:$PATH
export LD_LIBRARY_PATH=$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}
export CUDA_VISIBLE_DEVICES=0
cd $W/ESAM
$W/mc/envs/ESAM/bin/python $W/tools/distill/teacher.py \
  2>&1 | tee $W/logs/30_teacher.log
cat $W/distill/teacher/teacher_metrics.json | tail -40


In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
$W/mc/envs/ESAM/bin/python - <<'PY'
import json
m = json.load(open('/kaggle/working/distill/teacher/teacher_metrics.json'))
print(json.dumps(m['gate'], indent=2))
if not m['gate']['ok']:
    raise SystemExit('TEACHER_GATE_FAILED: inspect previews before training')
print('teacher gate OK')
PY


In [ ]:
%%bash
set +e
W=/kaggle/working
export PATH=$W/mc/envs/ESAM/bin:$PATH
export LD_LIBRARY_PATH=$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}
export CUDA_VISIBLE_DEVICES=0
export ESAM_FREEZE_SHARED=0
export ESAM_TARGET=teacher
export ESAM_MAX_MINUTES=110
export ESAM_LR=0.001
export ESAM_STUDENT=$W/distill/student_all
cd $W/ESAM
$W/mc/envs/ESAM/bin/python $W/tools/distill/student.py \
  2>&1 | tee $W/logs/40_student.log
ls -lh $W/distill/student_all 2>/dev/null | tail -30


In [ ]:
%%bash
set +e
W=/kaggle/working
echo "================ FINAL STATUS ================"
$W/mc/envs/ESAM/bin/python - <<'PY'
import json, os, glob
W = '/kaggle/working'
paths = [f'{W}/distill/teacher/teacher_metrics.json']
paths += sorted(glob.glob(f'{W}/distill/student*/student_ap.json'))
paths += sorted(glob.glob(f'{W}/distill/student*/distill_metrics.json'))
for p in paths:
    print('\n---', p, '---')
    if os.path.isfile(p):
        d = json.load(open(p))
        d.pop('history', None)
        print(json.dumps(d, indent=2)[:3000])
    else:
        print('MISSING')
for root, dirs, files in os.walk(f'{W}/distill'):
    if root.endswith('/frames') or root.endswith('/teacher'):
        continue
    for f in sorted(files):
        q = os.path.join(root, f)
        print(f'{os.path.getsize(q)/1e6:8.2f} MB  {os.path.relpath(q, W)}')
PY
df -h $W | tail -1
